# AIML 420 - Assignment 3

* author: Jason Pollock
* email: jason@pollock.ca
* email: pollocjaso@myvuw.ac.nz

## Part 1: Job Scheduling

Dependencies:

Specifying the dependencies makefile style

* Oj2: Oj1
* tn: tn-1 - This one is rather important, since it sets a floor on start time.
* O11: t=0
* O21: t=10
* O22: t=20

* O11: M1
* O12: M2
* O21: M1
* O22: M1
* O31: M1
* O32: M2

Process order:

* T1 = Process(O11, M1, t1, 50)
* T2 = Process(O21, M2, t2, 30)
* T3 = Process(O31, M1, t3, 40)
* T4 = Process(O12, M2, t4, 25)
* T5 = Process(O22, M1, t5, 35)
* T6 = Process(O32, M2, t6, 20)

Easiest to implement in code, so do that.

### Question 1: Earliest Starting Time

In [21]:
m1_next_available = 0
m2_next_available = 0

t1 = max(0, 0, m1_next_available)
m1_next_available = t1 + 50
t1_completion = m1_next_available

t2 = max(t1+1, 10, m2_next_available)
m2_next_available = t2 + 30
t2_completion = m2_next_available

t3 = max(t2+1, 20, m1_next_available)
m1_next_available = t3 + 40
t3_completion = m1_next_available

t4 = max(t3+1, t1+50, m2_next_available)
m2_next_available = t4 + 25
t4_completion = m2_next_available

t5 = max(t4+1, t2+30, m1_next_available)
m1_next_available = t5 + 35
t5_completion = m1_next_available

t6 = max(t5+1, t3+40, t4+25)
m2_next_available = t6 + 20
t6_completion = m2_next_available

print("Operation start times:")
print(f"T1 start: {t1}")
print(f"T2 start: {t2}")
print(f"T3 start: {t3}")
print(f"T4 start: {t4}")
print(f"T5 start: {t5}")
print(f"T6 start: {t6}")


T1 start: 0
T2 start: 10
T3 start: 50
T4 start: 51
T5 start: 90
T6 start: 91


### Question 2 Completion Time and Makespan

In [22]:
print("Job Completion Times")
print(f"J1: {t4_completion}")
print(f"J2: {t5_completion}")
print(f"J3: {t6_completion}")

print(f"Makespan: {max(t4_completion, t5_completion, t6_completion)}")

Job Completion Times
J1: 76
J2: 125
J3: 111
Makespan: 125


### Question 3: Shortest Processing Time SPT Dispatch

Assumption: The machines are not interchangeable, so the only thing changing is the T1<T2<T3 requirement.

Solving in code, there would be 3 heaps, pending, ready_machine1 and ready_machine2. At each time step, check if any tasks become ready, moving them to the heap. When the timestamp exceeds a machine's completion time, pop the top off the heap and start running it.

Could enhance this for "real" times by taking the max of the pending, and machine execution timestamps and processing that event.

The task flow below was achieved by running that algorithm iteratively.



In [24]:
def process(start, length, machine_next_available):
    task_start = max(start, machine_next_available)
    task_end = task_start + length
    return (task_start, task_end, task_end)

m1_next_available = 0
m2_next_available = 0


print("The processing order was:")
t1_start, t1_end, m1_next_available = process(0, 50, m1_next_available) # O11
print(f"process(O11, M1, {t1_start})")
t2_start, t2_end, m2_next_available = process(10,30, m2_next_available) # O21
print(f"process(O21, M2, {t2_start})")
t3_start, t3_end, m2_next_available = process(t1_end, 25, m2_next_available) # O12
print(f"process(O12, M2, {t3_start})")
t4_start, t4_end, m1_next_available = process(t2_end, 35, m1_next_available) # O22
print(f"process(O22, M1, {t4_start})")
t5_start, t5_end, m1_next_available = process(20, 40, m1_next_available) # O31
print(f"process(O31, M1, {t5_start})")
t6_start, t6_end, m2_next_available = process(t5_end, 20, m2_next_available) # O31
print(f"process(O32, M2, {t6_start})")


print("\n\nJob Completion Times")
print(f"J1: {t3_end}")
print(f"J2: {t4_end}")
print(f"J3: {t6_end}")

print(f"Makespan: {max(t3_end, t4_end, t6_end)}")

The processing order was:
process(O11, M1, 0)
process(O21, M2, 10)
process(O12, M2, 50)
process(O22, M1, 50)
process(O31, M1, 85)
process(O32, M2, 125)


Job Completion Times
J1: 75
J2: 85
J3: 145
Makespan: 145


### Question 5: SPT vs FSFS Comparison

The orderings are substantially different, as is the completion time.

This doesn't mean that one algorithm is better than the other. A different arrival time would have different results.

SPT would work best when the tasks are all of similar effort, and the availability of machines is high - overload (not having an available machine) should be rare. Starving expensive tasks is a pretty standard response to overload conditions, it is best to reject expensive work to process more total requests.

FCFS is a good general choice, but not good under overload. Jobs typically have a maximum allowed completion time, and FCFS will allow _all_ jobs to exceed that time when overloaded. SPT will sacrifice one job to make the most progress through the queue.

Typically, I would recommend using FCFS until the system becomes overloaded (unable to meet service agreements), then switch to SPT for the overloaded machines.

### Question 6: Alternative methods


#### Static backlog

If the work is static, then the problem is a search problem.

We can map this to a graph search:

1) making each operation a node
2) connect each node to every other node -> (starting bidirectional)
3) remove backlinks from dependent nodes (delete O22 -> O21)
4) every edge has a cost representing the next node's start time (remaining cost of parent task + remaining time to machine availability)
5) Topographically sort the tree to find the root operations (O11, O21, O31)
6) Run A* on the graph, where

* g(x) is current makespan (including machine idleness)
* h(x) is the minimum workspan of the remaining nodes, assuming full utilization of each machine - the makespan's lower bound.

Note: The edge cost for g(n) will need to be recalculated to obtain the new start time each iteration.
Note: Since the cost of a transition depends on the history of the path, it can't be cached the same way A* does Euclidean maps. The A* search is walking the task graph, but searching the set of process orderings.

Since f(x) = g(x) + h(x), the algorithm is optimizing workspan.

#### Dynamic backlog

Here we need to define a fitness function. While the fitness function for static backlog was workspan, with genetic programming we can optimize for more things, or even combine fitness functions.

Examples:
* fitness = min idle workers     (min cost)
* fitness = min total execution  (workspan)
* fitness = total value realized (overload)

This will propose to extend the problem to support multiple queues, but will not consider fitness functions other than workspan. This is probably only necessary if machines have capabilities that can do multiple non-overlapping types of tasks. If a machine is interchangeable (same capabilities, same location), it can be represented by a single queue and the problem collapses to sequencing.

The genetic programming solutions in the course separated routing from sequencing, first deciding where a task should go and then using another algorithm to learn task orderings on a machine.

The problem would be defined as a routing rule and a sequencing rule. The routing rule would score each machine's queue, with the machine reporting the highest score being selected. When a machine becomes "free", a sequencing rule is run on that machine's controlling queue, with the highest scored operation being selected.

The initial variables would be:

* m_wait_t - remaining wait time on the machine
* q_size - size of the machine's backlog
* q_length_t - total time of operations in queue
* op_wait_t - time required to run _this_ operation
* job_length_t - total time remaining for _this_ job
* job_length_op - total operations remaining for _this_ job

The operators would be:

* math - *,/,+,-
* ordering min, max

A member of the population is the {routing, sequencing} pair. Fitness will be measured against a set of recorded job arrivals and dependencies, replaying them against the rules and measuring workspan. Then standard crossover would be used, swapping subtrees with other matching rules. Crossover should also have the possibility of swapping the entire rule with another pair. Mutation would have the possibility of swapping the operation to another operation. This is safe because the operators all accept two numeric values and return a numeric value.


## Neural Networks

* Implement: Perceptron
* 3 features - attrib1, attrib2, attrib3
* training/test set already done
* Single file - perceptron.
